# Classificacao de Cancer de Pulmao com CNN

Este notebook usa o dataset **IQ-OTH/NCCD** em JPG para treinar uma CNN simples em `TensorFlow/Keras`.

## Objetivo
- entrada: uma imagem de slice de CT em formato JPG
- classes de origem: `Normal cases`, `Bengin cases`, `Malignant cases`
- alvo de treino: `0 = nao_maligno`, `1 = maligno`
- saida final: probabilidade de malignidade da imagem

## Limitacao Importante
A exportacao em JPG deste dataset nao preserva identificadores de paciente ou exame.
Por isso, o split deste notebook e **por imagem**, e nao por paciente.
Para fins academicos isso ainda e util, mas e uma avaliacao mais fraca do que um split por paciente.



In [ ]:
from pathlib import Path
import json
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tqdm import tqdm
from sklearn.metrics import (
confusion_matrix,
roc_curve,
roc_auc_score,
precision_recall_curve,
average_precision_score,
)

        ## Funcoes Auxiliares

        Estas funcoes leem o dataset, montam os splits e carregam as imagens.
        


In [ ]:
        PASTAS_CLASSES = {
            "Normal cases": 0,
            "Bengin cases": 0,
            "Malignant cases": 1,
        }

        NOMES_CLASSES = {
            0: "nao_maligno",
            1: "maligno",
        }


        def definir_semente(seed=42):
            random.seed(seed)
            np.random.seed(seed)
            tf.random.set_seed(seed)


        def coletar_registros_imagens(dataset_dir):
            dataset_dir = Path(dataset_dir)
            linhas = []
            for nome_pasta, rotulo_binario in PASTAS_CLASSES.items():
                pasta = dataset_dir / nome_pasta
                if not pasta.exists():
                    print(f"Aviso: pasta nao encontrada: {pasta}")
                    continue
                for image_path in sorted(pasta.glob("*.jpg")):
                    linhas.append(
                        {
                            "image_path": str(image_path),
                            "classe_original": nome_pasta,
                            "label": rotulo_binario,
                        }
                    )
            if not linhas:
                raise FileNotFoundError(f"Nenhuma imagem JPG foi encontrada em {dataset_dir}")
            frame = pd.DataFrame(linhas)
            return frame.sample(frac=1.0, random_state=42).reset_index(drop=True)


        def resumir_dataset(registros):
            print("Contagem por rotulo binario:")
            print(registros["label"].value_counts().sort_index().rename(index=NOMES_CLASSES))
            print()
            print("Contagem por pasta original:")
            print(registros["classe_original"].value_counts().sort_index())


        def split_estratificado(registros, seed=42, val_fraction=0.15, test_fraction=0.15):
            rng = random.Random(seed)
            partes_treino = []
            partes_val = []
            partes_teste = []

            for _, grupo in registros.groupby("label"):
                indices = list(grupo.index)
                rng.shuffle(indices)
                total = len(indices)

                test_count = max(1, int(round(total * test_fraction)))
                val_count = max(1, int(round(total * val_fraction)))
                if test_count + val_count >= total:
                    test_count = 1
                    val_count = 1

                test_idx = indices[:test_count]
                val_idx = indices[test_count:test_count + val_count]
                train_idx = indices[test_count + val_count:]

                partes_treino.append(registros.loc[train_idx])
                partes_val.append(registros.loc[val_idx])
                partes_teste.append(registros.loc[test_idx])

            treino_df = pd.concat(partes_treino).sample(frac=1.0, random_state=seed).reset_index(drop=True)
            val_df = pd.concat(partes_val).sample(frac=1.0, random_state=seed).reset_index(drop=True)
            teste_df = pd.concat(partes_teste).sample(frac=1.0, random_state=seed).reset_index(drop=True)
            return treino_df, val_df, teste_df


        def pegar_subconjunto_pequeno(registros, por_classe_original=30, seed=42):
            partes = []
            for _, grupo in registros.groupby("classe_original"):
                partes.append(grupo.sample(n=min(por_classe_original, len(grupo)), random_state=seed))
            return pd.concat(partes).sample(frac=1.0, random_state=seed).reset_index(drop=True)


        def carregar_tensor_imagem(image_path, image_size=224):
            image = keras.utils.load_img(
                image_path,
                color_mode="grayscale",
                target_size=(image_size, image_size),
            )
            array = keras.utils.img_to_array(image).astype("float32") / 255.0
            return array


        def montar_dataset_numpy(registros, image_size=224):
            imagens = []
            labels = []
            for row in tqdm(registros.itertuples(index=False), total=len(registros), desc="Carregando imagens"):
                imagens.append(carregar_tensor_imagem(row.image_path, image_size=image_size))
                labels.append(float(row.label))
            x = np.asarray(imagens, dtype=np.float32)
            y = np.asarray(labels, dtype=np.float32)
            return x, y


        def mostrar_exemplos(registros, amostras_por_classe=3, image_size=224):
            fig, axes = plt.subplots(len(PASTAS_CLASSES), amostras_por_classe, figsize=(4 * amostras_por_classe, 9))
            if amostras_por_classe == 1:
                axes = np.expand_dims(axes, axis=1)

            for row_index, nome_pasta in enumerate(PASTAS_CLASSES):
                subconjunto = registros[registros["classe_original"] == nome_pasta].head(amostras_por_classe)
                for col_index in range(amostras_por_classe):
                    ax = axes[row_index, col_index]
                    ax.axis("off")
                    if col_index >= len(subconjunto):
                        continue
                    image_path = subconjunto.iloc[col_index]["image_path"]
                    image = keras.utils.load_img(
                        image_path,
                        color_mode="grayscale",
                        target_size=(image_size, image_size),
                    )
                    ax.imshow(image, cmap="gray")
                    ax.set_title(nome_pasta)

            plt.tight_layout()
            plt.show()
        


        ## Modelo

        Esta e uma CNN pequena e facil de explicar em sala de aula.
        


In [ ]:
        def construir_modelo(image_size=224):
            aumento_dados = keras.Sequential(
                [
                    keras.layers.RandomFlip("horizontal"),
                    keras.layers.RandomRotation(0.05),
                    keras.layers.RandomZoom(0.1),
                ],
                name="aumento_dados",
            )

            model = keras.Sequential(
                [
                    keras.layers.Input(shape=(image_size, image_size, 1)),
                    aumento_dados,
                    keras.layers.Conv2D(16, 3, activation="relu", padding="same"),
                    keras.layers.MaxPooling2D(),
                    keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
                    keras.layers.MaxPooling2D(),
                    keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
                    keras.layers.MaxPooling2D(),
                    keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
                    keras.layers.GlobalAveragePooling2D(),
                    keras.layers.Dropout(0.3),
                    keras.layers.Dense(1, activation="sigmoid"),
                ]
            )
            return model
        


        ## Funcoes de Treino e Avaliacao

        Esta secao prepara os dados, treina o modelo e calcula metricas no conjunto de teste.
        


In [ ]:
        def calcular_metricas_binarias(y_true, y_prob, threshold=0.5):
            y_true = np.asarray(y_true).astype(np.int32)
            y_prob = np.asarray(y_prob).astype(np.float32)
            y_pred = (y_prob >= threshold).astype(np.int32)

            tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

            accuracy = float((tp + tn) / max(len(y_true), 1))
            precision = float(tp / max(tp + fp, 1))
            recall = float(tp / max(tp + fn, 1))
            specificity = float(tn / max(tn + fp, 1))

            return {
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "specificity": specificity,
                "tp": int(tp),
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
            }


        def escolher_melhor_limiar(y_true, y_prob):
            y_true = np.asarray(y_true).astype(np.int32)
            y_prob = np.asarray(y_prob).astype(np.float32)
            candidatos = np.linspace(0.1, 0.9, 81)

            melhor_threshold = 0.5
            melhor_metricas = None
            melhor_score = -1.0

            for threshold in candidatos:
                metricas = calcular_metricas_binarias(y_true, y_prob, threshold=threshold)
                score = metricas["recall"] + metricas["specificity"]
                if score > melhor_score:
                    melhor_score = score
                    melhor_threshold = float(threshold)
                    melhor_metricas = metricas

            return melhor_threshold, melhor_metricas


        def avaliar_probabilidades(y_true, y_prob, threshold=0.5):
            metricas = calcular_metricas_binarias(y_true, y_prob, threshold=threshold)
            y_true = np.asarray(y_true).astype(np.int32)
            y_prob = np.asarray(y_prob).astype(np.float32)

            try:
                auc_roc = float(roc_auc_score(y_true, y_prob))
            except ValueError:
                auc_roc = None

            try:
                avg_precision = float(average_precision_score(y_true, y_prob))
            except ValueError:
                avg_precision = None

            metricas["auc_roc"] = auc_roc
            metricas["average_precision"] = avg_precision
            metricas["threshold"] = float(threshold)
            return metricas


        def treinar_modelo(
            dataset_dir,
            output_dir,
            epochs=10,
            batch_size=16,
            lr=1e-3,
            image_size=224,
            val_fraction=0.15,
            test_fraction=0.15,
            seed=42,
            usar_subconjunto_pequeno=False,
            por_classe_subset=30,
        ):
            definir_semente(seed)
            output_dir = Path(output_dir)
            output_dir.mkdir(parents=True, exist_ok=True)

            registros = coletar_registros_imagens(dataset_dir)
            if usar_subconjunto_pequeno:
                registros = pegar_subconjunto_pequeno(
                    registros,
                    por_classe_original=por_classe_subset,
                    seed=seed,
                )

            resumir_dataset(registros)
            registros.to_csv(output_dir / "all_records.csv", index=False)

            treino_df, val_df, teste_df = split_estratificado(
                registros,
                seed=seed,
                val_fraction=val_fraction,
                test_fraction=test_fraction,
            )
            treino_df.to_csv(output_dir / "train_records.csv", index=False)
            val_df.to_csv(output_dir / "val_records.csv", index=False)
            teste_df.to_csv(output_dir / "test_records.csv", index=False)

            x_train, y_train = montar_dataset_numpy(treino_df, image_size=image_size)
            x_val, y_val = montar_dataset_numpy(val_df, image_size=image_size)
            x_test, y_test = montar_dataset_numpy(teste_df, image_size=image_size)

            model = construir_modelo(image_size=image_size)
            model.compile(
                optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                loss="binary_crossentropy",
                metrics=["accuracy"],
            )

            checkpoint_path = output_dir / "best.keras"
            callbacks = [
                keras.callbacks.ModelCheckpoint(
                    filepath=str(checkpoint_path),
                    monitor="val_accuracy",
                    mode="max",
                    save_best_only=True,
                ),
                keras.callbacks.EarlyStopping(
                    monitor="val_accuracy",
                    mode="max",
                    patience=3,
                    restore_best_weights=True,
                ),
            ]

            history = model.fit(
                x_train,
                y_train,
                validation_data=(x_val, y_val),
                epochs=epochs,
                batch_size=batch_size,
                callbacks=callbacks,
                verbose=1,
            )

            melhor_modelo = keras.models.load_model(checkpoint_path)

            y_prob_val = melhor_modelo.predict(x_val, verbose=0).ravel()
            melhor_threshold, metricas_val = escolher_melhor_limiar(y_val, y_prob_val)

            y_prob_test = melhor_modelo.predict(x_test, verbose=0).ravel()
            metricas_teste = avaliar_probabilidades(y_test, y_prob_test, threshold=melhor_threshold)

            metadata = {
                "image_size": image_size,
                "class_names": NOMES_CLASSES,
                "positive_class": "maligno",
                "negative_class": "nao_maligno",
                "dataset_dir": str(dataset_dir),
                "used_small_subset": bool(usar_subconjunto_pequeno),
                "per_class_subset": int(por_classe_subset),
                "best_threshold": float(melhor_threshold),
            }

            with (output_dir / "best_info.json").open("w", encoding="utf-8") as file:
                json.dump(metadata, file, indent=2)

            with (output_dir / "metrics.json").open("w", encoding="utf-8") as file:
                json.dump(
                    {
                        "history": history.history,
                        "validation_metrics": metricas_val,
                        "test_metrics": metricas_teste,
                        "train_images": len(treino_df),
                        "val_images": len(val_df),
                        "test_images": len(teste_df),
                        **metadata,
                    },
                    file,
                    indent=2,
                )

            print("Modelo salvo em:", checkpoint_path)
            print("Melhor limiar encontrado na validacao:", round(melhor_threshold, 4))
            print("Metricas de teste:", metricas_teste)

            resultados = {
                "treino_df": treino_df,
                "val_df": val_df,
                "teste_df": teste_df,
                "x_test": x_test,
                "y_test": y_test,
                "y_prob_test": y_prob_test,
                "melhor_threshold": melhor_threshold,
                "metricas_teste": metricas_teste,
            }
            return melhor_modelo, history.history, resultados


        def plotar_curvas_treino(history):
            if not history:
                print("Historico vazio.")
                return

            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].plot(history.get("loss", []), label="treino")
            axes[0].plot(history.get("val_loss", []), label="validacao")
            axes[0].set_title("Loss")
            axes[0].legend()

            axes[1].plot(history.get("accuracy", []), label="treino")
            axes[1].plot(history.get("val_accuracy", []), label="validacao")
            axes[1].set_title("Accuracy")
            axes[1].legend()

            plt.tight_layout()
            plt.show()


        def plotar_matriz_confusao(y_true, y_prob, threshold=0.5):
            y_true = np.asarray(y_true).astype(np.int32)
            y_pred = (np.asarray(y_prob) >= threshold).astype(np.int32)
            matriz = confusion_matrix(y_true, y_pred, labels=[0, 1])

            fig, ax = plt.subplots(figsize=(5, 4))
            im = ax.imshow(matriz, cmap="Blues")
            ax.set_title("Matriz de Confusao")
            ax.set_xlabel("Predito")
            ax.set_ylabel("Real")
            ax.set_xticks([0, 1], ["nao_maligno", "maligno"])
            ax.set_yticks([0, 1], ["nao_maligno", "maligno"])

            for i in range(matriz.shape[0]):
                for j in range(matriz.shape[1]):
                    ax.text(j, i, str(matriz[i, j]), ha="center", va="center", color="black")

            fig.colorbar(im, ax=ax)
            plt.tight_layout()
            plt.show()


        def plotar_curva_roc(y_true, y_prob):
            y_true = np.asarray(y_true).astype(np.int32)
            y_prob = np.asarray(y_prob).astype(np.float32)
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            auc = roc_auc_score(y_true, y_prob)

            plt.figure(figsize=(5, 4))
            plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
            plt.plot([0, 1], [0, 1], "--", color="gray")
            plt.xlabel("False Positive Rate")
            plt.ylabel("True Positive Rate")
            plt.title("Curva ROC")
            plt.legend()
            plt.tight_layout()
            plt.show()


        def plotar_curva_precision_recall(y_true, y_prob):
            y_true = np.asarray(y_true).astype(np.int32)
            y_prob = np.asarray(y_prob).astype(np.float32)
            precision, recall, _ = precision_recall_curve(y_true, y_prob)
            ap = average_precision_score(y_true, y_prob)

            plt.figure(figsize=(5, 4))
            plt.plot(recall, precision, label=f"AP = {ap:.4f}")
            plt.xlabel("Recall")
            plt.ylabel("Precision")
            plt.title("Curva Precision-Recall")
            plt.legend()
            plt.tight_layout()
            plt.show()
        


        ## Caminhos e Configuracoes

        Atualize estes valores se o dataset estiver em outro local.
        


In [ ]:
DATASET_DIR = Path(r"C:/Users/Usuario/Documents/Lung Cancer/The IQ-OTHNCCD lung cancer dataset/The IQ-OTHNCCD lung cancer dataset/The IQ-OTHNCCD lung cancer dataset/The IQ-OTHNCCD lung cancer dataset")
OUTPUT_DIR = Path(r"c:/Coding/studies/python/computer_vision/lung_cancer_detection/iqoth_output")

IMAGE_SIZE = 224
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
SEED = 42

USAR_SUBCONJUNTO_PEQUENO = True
POR_CLASSE_SUBSET = 30
        


        ## Explorar o Dataset
        


In [ ]:
registros = coletar_registros_imagens(DATASET_DIR)
resumir_dataset(registros)
mostrar_exemplos(registros, amostras_por_classe=3, image_size=IMAGE_SIZE)

        ## Treino

        Para um teste rapido, mantenha `USAR_SUBCONJUNTO_PEQUENO = True`.

        Para o treino completo, altere para:

        ```python
        USAR_SUBCONJUNTO_PEQUENO = False
        ```
        


In [ ]:
# Descomente para treinar.
# modelo, history, resultados = treinar_modelo(
#     dataset_dir=DATASET_DIR,
#     output_dir=OUTPUT_DIR,
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     lr=LEARNING_RATE,
#     image_size=IMAGE_SIZE,
#     seed=SEED,
#     usar_subconjunto_pequeno=USAR_SUBCONJUNTO_PEQUENO,
#     por_classe_subset=POR_CLASSE_SUBSET,
# )
# plotar_curvas_treino(history)



        ## Avaliacao

        Execute estas celulas depois do treino para visualizar as metricas finais.
        


In [ ]:
        # Exemplo de uso depois do treino:
        # print(resultados["metricas_teste"])
        # plotar_matriz_confusao(
        #     resultados["y_test"],
        #     resultados["y_prob_test"],
        #     threshold=resultados["melhor_threshold"],
        # )
        # plotar_curva_roc(resultados["y_test"], resultados["y_prob_test"])
        # plotar_curva_precision_recall(resultados["y_test"], resultados["y_prob_test"])
        


        ## Inferencia

        Esta funcao recebe uma imagem JPG inteira e devolve a probabilidade de malignidade.
        


In [ ]:
def prever_imagem(checkpoint_path, image_path, image_size=None, threshold=None):
            checkpoint_path = Path(checkpoint_path)
            image_path = Path(image_path)
            metadata_path = checkpoint_path.with_name("best_info.json")
            metadata = {}

            if metadata_path.exists():
                with metadata_path.open("r", encoding="utf-8") as file:
                    metadata = json.load(file)

            image_size = image_size or metadata.get("image_size", 224)
            threshold = threshold if threshold is not None else metadata.get("best_threshold", 0.5)

            model = keras.models.load_model(checkpoint_path)
            image = carregar_tensor_imagem(str(image_path), image_size=image_size)
            probability = float(model.predict(image[np.newaxis, ...], verbose=0)[0][0])
            prediction = "maligno" if probability >= threshold else "nao_maligno"

            print(f"probabilidade_malignidade={probability:.4f}")
            print(f"limiar_utilizado={float(threshold):.4f}")
            print(f"classe_predita={prediction}")
            return probability, prediction
        


In [ ]:
        # Exemplo de inferencia apos o treino:
        # probability, prediction = prever_imagem(
        #     checkpoint_path=OUTPUT_DIR / "best.keras",
        #     image_path=DATASET_DIR / "Malignant cases" / "Malignant case (1).jpg",
        # )
        
